# 05 · Clearance maps

Asking MapRT for a clearance map, and reading it.

> **Teaching use only. Not a medical device.** Every call in this workshop goes to an emulator on your own machine, answering from sessions recorded on a real MapRT appliance and anonymized. Not affiliated with or endorsed by Vision RT.

In [ ]:
# Setup. Run this cell first, in every notebook.
# On Google Colab it installs the workshop, downloads the session packs and starts the emulator
# (about a minute, the first time). Anywhere else it does nothing: that was done for you.
import os, subprocess, sys

if "COLAB_RELEASE_TAG" in os.environ:
    workshop = os.environ.get("SGRT_WORKSHOP_DIR", "/content/sgrt-scripting-uk-2026")
    if not os.path.isdir(workshop):
        repo = os.environ.get("SGRT_WORKSHOP_REPO", "https://github.com/tallhamer/sgrt-scripting-uk-2026")
        ref = os.environ.get("SGRT_WORKSHOP_REF", "v2026.12.0")
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", ref, repo, workshop], check=True)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", workshop], check=True)
    os.chdir(f"{workshop}/notebooks")
    subprocess.run([sys.executable, "-m", "sgrt_workshop.packs"], check=True)
    subprocess.run([sys.executable, "-m", "sgrt_workshop.emulator", "start"], check=True)

print("Ready.")

## What is a clearance map?

For a single *setup* (that is, a combination of patient surface, treatment room geometry,
isocenter location, patient position, and safety margins), MapRT calculates a **clearance map**. A
**clearance map** is an evaluation of the clearance state for every possible combination of couch
and gantry angle: *clear*, or a *collision* (the gantry would come too close to the patient or the
couch). The results are placed in a 2D grid, with gantry angle on the vertical axis and couch angle
on the horizontal axis, giving the user a *map* of the solution space.

To ask for one, MapRT needs the setup:

* the **patient surface** (notebook 04) and the **treatment room** (notebook 03);
* the **isocenter**, in millimeters, and the **patient position** (head first supine, `HFS`...);
* two safety margins, the **couch buffer** and the **patient buffer**: how close the gantry may
  come, in centimeters.

The **isocenter** and **patient position** come from the treatment plan.

## The plan and structure set

The session pack for `ANON-001-SLIM` includes the patient's **RT Plan** and **RT Structure Set**, as
anonymized DICOM files. This cell unpacks them and prints where they are:

In [ ]:
from sgrt_workshop import packs

dicom = packs.extract_dicom("hfs-phantom-slim")

print("RT Plan:         ", dicom.plan)
print("RT Structure Set:", dicom.struct)

**You don't need to find or open these files yourself**: the variables `dicom.plan` and
`dicom.struct` hold their full paths, and the code below uses them. The file names are long
because DICOM files are named by their unique ID.

If you want to look at them:

* **On your own laptop:** they are in the workshop folder, under
  `packs/hfs-phantom-slim-dicom/plan/` and `packs/hfs-phantom-slim-dicom/struct/`. You can open them in any DICOM
  viewer, or use these paths in your own scripts after the course.
* **On Binder:** in the file panel on the left, open `packs`, then `hfs-phantom-slim-dicom`.
* **On Colab:** in the **Files** panel on the left, open `sgrt-scripting-uk-2026`, `packs`, then
  `hfs-phantom-slim-dicom`.

On Binder and Colab the files live only as long as your session.

## The setup, from the plan

`pydicom` reads DICOM files. The isocenter is stored with each beam (every beam of this plan shares
it); we round it to 0.01 mm, the precision MapRT works to. The patient position is in the plan's
setup section (`+ 0.0` just turns a rounded `-0.0` into a plain `0.0`):

In [ ]:
import pydicom

plan = pydicom.dcmread(dicom.plan)

isocenter = [round(float(v), 2) + 0.0 for v in plan.BeamSequence[0].ControlPointSequence[0].IsocenterPosition]
position = plan.PatientSetupSequence[0].PatientPosition

print("Patient:   ", plan.PatientID)
print("Isocenter: ", isocenter, "mm")
print("Position:  ", position)

The patient's surface and the room come from MapRT:

In [ ]:
from sgrt_workshop import connect

client = connect()
room_ids = {name: room_id for name, room_id, _ in client.get_treatment_rooms()}
surface_id, _ = client.get_patient_surfaces(plan.PatientID)[0]

room = "Truebeam"
print(room, room_ids[room])
print("surface", surface_id)

## Building the API request

`build_map_request` turns the setup information into a MapRT request body. MapRT can compute a
clearance map at **low resolution** (every 5 to 10 degrees, quick) or **high resolution** (every
degree); this gives you both requests, and a short label describing the setup:

In [ ]:
import json

from maprt_client import build_map_request

label, (low_res, high_res) = build_map_request(
    isocenter=isocenter,
    couch_buffer=2,      # cm
    patient_buffer=2,    # cm
    surface_id=surface_id,
    room_id=room_ids[room],
    room_scale="IEC_61217",
    patient_orientation=position,
)

print("Label:", label)
print()
print("Low-resolution request:")
print(json.dumps(low_res, indent=2))
print()
print("High-resolution request:")
print(json.dumps(high_res, indent=2))

The two requests differ only in `HighResolution`. Notice the buffers: you give centimeters, the
request carries millimeters (`20`).

## The reply is not JSON

GetMap is the one MapRT endpoint whose answer isn't the usual envelope. It is plain text, one line
per angle pair. `client.send` shows each reply exactly as it arrives; here are the start and the
end of both:

In [ ]:
raw_low = client.send("POST", "/integration/GetMap", low_res).text
raw_high = client.send("POST", "/integration/GetMap", high_res).text

for name, raw in (("Low-resolution", raw_low), ("High-resolution", raw_high)):
    lines = raw.split()
    print(f"{name} reply: {len(lines):,} lines")
    print("\n".join(lines[:5]))
    print("...")
    print("\n".join(lines[-3:]))
    print()

The first line gives the number of couch and gantry angles in the map: `37,36` at low resolution
(a couch angle every 5°, a gantry angle every 10°) and `181,360` at high resolution (every degree),
so the high-resolution reply is about 50 times longer. Each line after that is `couch,gantry,isOK`
(`1` clear, `0` collision), and the last line is `DONE`, which says the map is complete
(notebook 07 shows why that matters).

The lines don't arrive in the order the map is drawn in:

* **Couch:** for each gantry angle, the couch counts *down* from 90° to 0°, then jumps to 355° (359°
  at high resolution) and carries on down to 270°.
* **Gantry:** it starts at 180°, counts up to 359°, wraps to 0° and stops at 179°.

MapRT's own screen draws the map with gantry 180° at the top and couch 270° (= -90°) on the left,
through 0°, to 90° on the right. To get the same picture, the reply has to be turned back into a
grid with both axes put in that order, minding the jumps at couch 0° and gantry 0°. A
low-resolution map also has to be spread out to one value per degree, each sample covering the
angles up to the next one.

The client module's `get_map` method sends the same request and turns the text into a grid of
numbers for you, returning a numpy *array* for convenience:

In [ ]:
collision_map, _, _ = client.get_map(high_res)
low_res_map, _, _ = client.get_map(low_res)

print("High resolution: shape", collision_map.shape, f"{collision_map.mean():.0%} clear")
print("Low resolution:  shape", low_res_map.shape, f"{low_res_map.mean():.0%} clear")

Both have 360 rows, one per **gantry** angle from 180° around to 179°, and 181 columns, one per
**couch** angle from 270° (= -90°) through 0° to 90°. `get_map` spreads the low-resolution map out
to the same grid, so the same code reads either one.

## Drawing it

In [ ]:
from sgrt_workshop.plotting import plot_maps

plot_maps({
    f"High resolution ({room}, 2 cm buffers)": (collision_map, None),
    f"Low resolution ({room}, 2 cm buffers)": (low_res_map, None),
}, columns=2)

Pale is clear, red is a collision. **Hover over a map** to read the couch and gantry angle under
the pointer. The red regions are where the gantry would come too close to the patient or the couch
as it turns, at that couch angle. The low-resolution map is the same picture in coarser blocks (5°
of couch by 10° of gantry): it is quicker to compute, but its edges are less exact.

To look up one pair of angles yourself: row `(gantry - 180) % 360`, column `(couch - 270) % 360`
(`%` is the remainder after division, which wraps angles around the circle):

In [ ]:
gantry, couch = 90, 0
row, column = (gantry - 180) % 360, (couch - 270) % 360

print(f"gantry {gantry}°, couch {couch}°:", "clear" if collision_map[row, column] else "COLLISION")

## Bigger buffers, less room

A larger buffer keeps the gantry farther from the patient, so less of the map is clear. Here are
high-resolution maps with 2 cm and 3 cm buffers, side by side:

In [ ]:
maps = {}
for buffer in (2, 3):
    _, (_, request) = build_map_request(
        isocenter=isocenter,
        couch_buffer=buffer,      # cm
        patient_buffer=buffer,    # cm
        surface_id=surface_id,
        room_id=room_ids[room],
        room_scale="IEC_61217",
        patient_orientation=position,
    )
    grid, _, _ = client.get_map(request)
    maps[buffer] = grid
    print(f"{buffer} cm buffers: {grid.mean():.1%} clear")

plot_maps({
    "2 cm buffers": (maps[2], None),
    "3 cm buffers": (maps[3], None),
}, columns=2)

The two maps are hard to tell apart by eye. Because each map is a numpy array of `1`s and `0`s,
subtracting one from the other shows exactly where they differ: `1` where an angle pair is clear
with 2 cm buffers but a collision with 3 cm, `0` where nothing changed (and `-1` where only the
3 cm map is clear, which a bigger buffer shouldn't give):

In [ ]:
from sgrt_workshop.plotting import plot_difference

difference = maps[2] - maps[3]
changed = (difference != 0).sum()

print(f"{changed:,} angle pairs changed: {changed / difference.size:.1%} of the map")
plot_difference(difference, title="2 cm minus 3 cm buffers",
                labels=("clear at 2 cm only", "clear at 3 cm only"))

Orange marks the angle pairs that the 3 cm buffers take away; they lie along the edges of the
collision regions. The difference is often very small, as it is here. In some cases, though, a
small change in the buffers can make an otherwise passing beam fail: a beam whose arc runs close
to the edge of a collision region can be clear with 2 cm buffers and collide with 3 cm.
Notebook 06 checks a plan's beams against a map.

## Exercise (optional)

Write a function `is_clear(collision_map, gantry, couch)` that returns `True` if that pair of
angles is clear. Use it to check gantry 0°, 90°, 180° and 270°, all at couch 0°.

Write your code in the empty cell below. The solution is under it, folded away: open it when you're done, or if you get stuck. There's no need to finish this in class.

<details>
<summary><b>Solution</b> (click to open)</summary>

```python
def is_clear(collision_map, gantry, couch):
    return bool(collision_map[(gantry - 180) % 360, (couch - 270) % 360])

for gantry in (0, 90, 180, 270):
    print(f"gantry {gantry:3}°, couch 0°:", "clear" if is_clear(collision_map, gantry, 0) else "COLLISION")
```

</details>